In [1]:
import sys
import torch
import cace

/global/software/rocky-8.x86_64/manual/modules/langs/anaconda3/2024.02-1/lib/python3.11/site-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
root = './best_model.pth'
average_E0 = {1: -5.853064337340629, 8: -2.926532168670322}
cace_nnp = torch.load(root, map_location='cpu')

In [3]:
to_read = "test-H2O_RPBE-D3.xyz"
to_bec_test = "h2o_bec.xyz"

test_write = './cace_test.xyz'
bec_write= './cace_bec_test.xyz'
from ase.io import read


In [5]:
evaluator = cace.tasks.EvaluateTask(model_path=cace_nnp, device='cpu',
                                    energy_key = 'CACE_energy',
                                    forces_key = 'CACE_forces',
                                    other_keys=['q_eq'],
                                    atomic_energies = average_E0,
                                    )
from ase.io import read
dataset = read(to_read, index=':')
result = evaluator(data=dataset, batch_size=1, compute_stress=False, xyz_output=test_write)

In [7]:
cace_nnp = torch.load(root, map_location='cpu')
cace_representation = cace_nnp.representation
q = cace_nnp.output_modules[1]
ep = cace_nnp.output_modules[2]
polarization = cace.modules.Polarization(pbc=True,
                                        normalization_factor=1.333/9.48933,
                                        charge_key='q_eq')

print(polarization.normalization_factor)

grad  = cace.modules.Grad(
    y_key = 'polarization',
    x_key = 'positions',
    output_key = 'bec_complex'
)
dephase = cace.modules.Dephase(
    input_key = 'bec_complex',
    phase_key = 'phase',
    output_key = 'CACE_BEC'
)
cace_bec = cace.models.NeuralNetworkPotential(
    input_modules=None,
    representation=cace_representation,
    output_modules=[q, ep, polarization, grad, dephase],
)
evaluator = cace.tasks.EvaluateTask(model_path=cace_bec, device='cpu',
                                    other_keys=['q_eq', 'CACE_BEC'],
                                    )

0.14047356346549228


In [8]:
dataset = read(to_bec_test, index=':')
result = evaluator(data=dataset, batch_size=1, compute_stress=False, xyz_output=bec_write)

### metrics

In [9]:
import numpy as np
dataset = read(test_write, index=':')
# pred_energy = result['energy']    
# pred_forces = result['forces'] 
pred_energy = np.array([atoms.info['CACE_energy'] for atoms in dataset])
pred_forces = ref_forces = np.vstack([atoms.get_array('CACE_forces') for atoms in dataset])
ref_energy = np.array([atoms.info['energy'] for atoms in dataset])
ref_forces = np.vstack([atoms.get_array('forces') for atoms in dataset])

def rmse(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    return np.sqrt(np.mean((a - b)**2))
    
atoms_list = read(to_read, index=":")
n_atoms = np.array([len(atoms) for atoms in atoms_list])

ref_e_per_atom  = ref_energy  / n_atoms
pred_e_per_atom = pred_energy / n_atoms

rmse_per_atom = rmse(ref_e_per_atom, pred_e_per_atom)
energy_rmse = rmse(ref_energy, pred_energy)
force_rmse  = rmse(ref_forces, pred_forces)


# print(f"Energy RMSE: {energy_rmse:.4f} eV")
print(f"Per-atom energy RMSE: {rmse_per_atom*1000:.2f} meV/atom")
print(f"Force  RMSE: {force_rmse:.4f} eV/Å")

Per-atom energy RMSE: 0.21 meV/atom
Force  RMSE: 0.0216 eV/Å


In [13]:

cace_xyz = read(bec_write, index=':')

epsilon_r = 1.78 # float(sys.argv[2])

bec_refs = []
bec_preds = []
atomic_nums = []
for xyz in cace_xyz:
    bec_ref = xyz.get_array('BEC')
    bec_refs.append(bec_ref)
    bec_pred = xyz.get_array('CACE_BEC') * -1
    bec_preds.append(bec_pred)
    atomic_num = xyz.arrays['numbers']
    atomic_nums.append(atomic_num)

all_ref = np.concatenate([x.flatten() for x in bec_refs])
all_pred = np.concatenate([x.flatten() for x in bec_preds])
bec_errors = all_ref - all_pred

mae = np.mean(np.abs(all_ref - all_pred))
rmse = np.sqrt(np.mean((all_ref - all_pred)**2))
r2 = 1 - np.sum(bec_errors**2) / np.sum((all_ref - np.mean(all_ref))**2)

print('MAE: ', mae)
print('RMSE: ', rmse)
print('R2: ', r2)

MAE:  0.045492081130902785
RMSE:  0.06299378705647578
R2:  0.974945039040176


In [14]:
import numpy as np

def metrics(y, yhat):
    err = y - yhat
    mae  = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    denom = np.sum((y - np.mean(y))**2)
    r2 = 1.0 - np.sum(err**2)/denom if denom > 0 else np.nan  # 상수 배열 보호
    return mae, rmse, r2

# all
mae_all, rmse_all, r2_all = metrics(all_ref, all_pred)

idx = np.arange(all_ref.size) % 9
diag_mask    = np.isin(idx, [0, 4, 8])   # xx, yy, zz
offdiag_mask = ~diag_mask                # xy, xz, yx, yz, zx, zy

# calc
mae_diag, rmse_diag, r2_diag       = metrics(all_ref[diag_mask],    all_pred[diag_mask])
mae_offd, rmse_offd, r2_offd       = metrics(all_ref[offdiag_mask], all_pred[offdiag_mask])

print("=== BEC metrics ===")
print(f"ALL     : MAE={mae_all:.6f}  RMSE={rmse_all:.6f}  R2={r2_all:.6f}")
print(f"Diagonal: MAE={mae_diag:.6f} RMSE={rmse_diag:.6f} R2={r2_diag:.6f}")
print(f"OffDiag : MAE={mae_offd:.6f} RMSE={rmse_offd:.6f} R2={r2_offd:.6f}")

=== BEC metrics ===
ALL     : MAE=0.045492  RMSE=0.062994  R2=0.974945
Diagonal: MAE=0.063153 RMSE=0.084392 R2=0.983724
OffDiag : MAE=0.036662 RMSE=0.048901 R2=0.872636
